# Feature Engineering
Chapter 4 of the book: "Build Your Own AI Investor"

In [ ]:
from __future__ import annotations

import pandas as pd
from matplotlib import pyplot as plt
from scipy.stats import zscore
from sklearn.preprocessing import PowerTransformer

In [ ]:
plt.rcParams["figure.figsize"] = [7.0, 4.5]
plt.rcParams["figure.dpi"] = 150

## Configuration
Constants for file paths, NaN-fill columns, and ratio clipping bounds.

In [ ]:
# --- File paths -----------------------------------------------------------
FUNDAMENTALS_FILE = "Annual_Stock_Price_Fundamentals_Filtered.csv"
PERFORMANCE_FILE = "Annual_Stock_Price_Performance_Filtered.csv"
FUNDAMENTALS_2024_FILE = "Annual_Stock_Price_Fundamentals_Filtered_2024_present.csv"

OUTPUT_RATIOS_FILE = "Annual_Stock_Price_Fundamentals_Ratios.csv"
OUTPUT_PERF_FILE = "Annual_Stock_Price_Performance_Percentage.csv"
OUTPUT_RATIOS_2024_FILE = "Annual_Stock_Price_Fundamentals_Ratios_2024.csv"

# --- Columns whose NaNs should be filled with 0 --------------------------
NAN_FILL_COLUMNS: list[str] = [
    "Short Term Debt",
    "Long Term Debt",
    "Interest Expense, Net",
    "Income Tax (Expense) Benefit, Net",
    "Cash, Cash Equivalents & Short Term Investments",
    "Property, Plant & Equipment, Net",
    "Revenue",
    "Gross Profit",
    "Total Current Liabilities",
]

# --- Clipping bounds for each ratio (ratio_name -> (lower, upper)) --------
CLIP_BOUNDS: dict[str, tuple[float, float]] = {
    "EV/EBIT": (-500, 500),
    "Op. In./(NWC+FA)": (-5, 5),
    "P/E": (-1000, 1000),
    "P/B": (-50, 100),
    "P/S": (0, 500),
    "Op. In./Interest Expense": (-600, 600),
    "Working Capital Ratio": (0, 30),
    "RoE": (-5, 5),
    "ROCE": (-2, 2),
    "Debt/Equity": (0, 100),
    "Debt Ratio": (0, 50),
    "Cash Ratio": (0, 30),
    "Gross Profit Margin": (0, 1),
    "(CA-CL)/TA": (-1.5, 2),
    "RE/TA": (-20, 2),
    "EBIT/TA": (-2, 1),
    "Book Equity/TL": (-2, 20),
    "Asset Turnover": (-2000, 2000),
}

# Read in Stock Data from last notebook

In [ ]:
x_raw = pd.read_csv(FUNDAMENTALS_FILE, index_col=0)
y_raw = pd.read_csv(PERFORMANCE_FILE, index_col=0)
print(f"Fundamentals: {x_raw.shape}  |  Performance: {y_raw.shape}")

# Get Fundamental Ratios

In [ ]:
def fill_nan_columns(df: pd.DataFrame, columns: list[str] | None = None) -> pd.DataFrame:
    """Return a copy with specified columns' NaNs replaced by 0."""
    df = df.copy()
    columns = columns or NAN_FILL_COLUMNS
    present = [c for c in columns if c in df.columns]
    df[present] = df[present].fillna(0)
    return df


def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with Enterprise Value (EV) and EBIT columns added."""
    df = df.copy()
    df["EV"] = (
        df["Market Cap"]
        + df["Long Term Debt"]
        + df["Short Term Debt"]
        - df["Cash, Cash Equivalents & Short Term Investments"]
    )
    df["EBIT"] = (
        df["Net Income (Common)"]
        - df["Interest Expense, Net"]
        - df["Income Tax (Expense) Benefit, Net"]
    )
    return df

In [ ]:
def compute_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Derive financial ratios from raw fundamental data.

    Returns a new DataFrame whose columns are the ratio features.
    """
    ratios = pd.DataFrame(index=df.index)

    # Valuation ratios
    ratios["EV/EBIT"] = df["EV"] / df["EBIT"]
    ratios["P/E"] = df["Market Cap"] / df["Net Income (Common)"]
    ratios["P/B"] = df["Market Cap"] / df["Total Equity"]
    ratios["P/S"] = df["Market Cap"] / df["Revenue"]

    # Profitability ratios
    nwc_fa = (
        df["Total Current Assets"]
        - df["Total Current Liabilities"]
        + df["Property, Plant & Equipment, Net"]
    )
    ratios["Op. In./(NWC+FA)"] = df["Operating Income (Loss)"] / nwc_fa
    ratios["RoE"] = df["Net Income (Common)"] / df["Total Equity"]
    ratios["ROCE"] = df["EBIT"] / (df["Total Assets"] - df["Total Current Liabilities"])
    ratios["Gross Profit Margin"] = df["Gross Profit"] / df["Revenue"]

    # Leverage / solvency ratios
    ratios["Op. In./Interest Expense"] = df["Operating Income (Loss)"] / -df["Interest Expense, Net"]
    ratios["Debt/Equity"] = df["Total Liabilities"] / df["Total Equity"]
    ratios["Debt Ratio"] = df["Total Assets"] / df["Total Liabilities"]

    # Liquidity ratios
    ratios["Working Capital Ratio"] = df["Total Current Assets"] / df["Total Current Liabilities"]
    ratios["Cash Ratio"] = (
        df["Cash, Cash Equivalents & Short Term Investments"] / df["Total Current Liabilities"]
    )

    # Efficiency
    ratios["Asset Turnover"] = df["Revenue"] / df["Property, Plant & Equipment, Net"]

    # Altman Z-score components
    ratios["(CA-CL)/TA"] = (
        (df["Total Current Assets"] - df["Total Current Liabilities"]) / df["Total Assets"]
    )
    ratios["RE/TA"] = df["Retained Earnings"] / df["Total Assets"]
    ratios["EBIT/TA"] = df["EBIT"] / df["Total Assets"]
    ratios["Book Equity/TL"] = df["Total Equity"] / df["Total Liabilities"]

    return ratios.fillna(0)


def clip_ratios(
    ratios: pd.DataFrame,
    bounds: dict[str, tuple[float, float]] | None = None,
) -> pd.DataFrame:
    """Clip ratio distributions to configured bounds.

    Returns a new DataFrame with outliers winsorised.
    """
    bounds = bounds or CLIP_BOUNDS
    ratios = ratios.copy()
    for col, (lo, hi) in bounds.items():
        if col in ratios.columns:
            ratios[col] = ratios[col].clip(lo, hi)
    return ratios

In [ ]:
def compute_performance(y_raw: pd.DataFrame) -> pd.DataFrame:
    """Compute relative price change from raw price data."""
    return pd.DataFrame({
        "Ticker": y_raw["Ticker"],
        "Perf": ((y_raw["Open Price2"] - y_raw["Open Price"]) / y_raw["Open Price"]).fillna(0),
    })

In [ ]:
def clip_by_zscore(df: pd.DataFrame, threshold: float = 3.0) -> pd.DataFrame:
    """Winsorise each column so values beyond *threshold* z-scores
    are replaced by the value at that z-score boundary.

    Returns a new DataFrame.
    """
    df = df.copy()
    z_scores = df.apply(zscore, nan_policy="omit")
    for col in df.columns:
        upper = threshold * df[col].std() + df[col].mean()
        lower = -threshold * df[col].std() + df[col].mean()
        df.loc[z_scores[col] > threshold, col] = upper
        df.loc[z_scores[col] < -threshold, col] = lower
    return df

In [ ]:
def build_ratio_features(raw_fundamentals: pd.DataFrame) -> pd.DataFrame:
    """End-to-end pipeline: raw fundamentals -> cleaned, clipped ratios."""
    df = fill_nan_columns(raw_fundamentals)
    df = add_derived_columns(df)
    ratios = compute_ratios(df)
    return clip_ratios(ratios)

### Run the functions

In [ ]:
X = build_ratio_features(x_raw)
y = compute_performance(y_raw)

print(f"X shape: {X.shape}  |  y shape: {y.shape}")

In [ ]:
X

### Before

In [ ]:
x_raw.head()

In [ ]:
y_raw.head()

### After

In [ ]:
X.head() # see X

In [ ]:
y.head() # see y

### See Distributions

In [ ]:
ratio_idx = 2  # change to explore different ratios (0 .. len(X.columns)-1)
col = X.columns[ratio_idx]
X[col].hist(bins=100, figsize=(6, 5))
plt.title(col);

In [ ]:
X.describe()

In [ ]:
def plot_all_distributions(df: pd.DataFrame, bins: int = 100, cols: int = 3) -> None:
    """Plot histograms for every column in *df*."""
    n = len(df.columns)
    rows = -(-n // cols)  # ceiling division
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = axes.flatten()
    for i, col_name in enumerate(df.columns):
        df[col_name].hist(bins=bins, ax=axes[i])
        axes[i].set_title(col_name)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    fig.tight_layout()

plot_all_distributions(X)

In [ ]:
y.to_csv(OUTPUT_PERF_FILE)
X.to_csv(OUTPUT_RATIOS_FILE)
print(f"Saved {OUTPUT_PERF_FILE} and {OUTPUT_RATIOS_FILE}")

### Try out power transformer see if our data has good distributions
A lot of the algorithms won't work without appropriate transformation. We'll use the power transformer

In [ ]:
def plot_before_after_transform(
    original: pd.DataFrame,
    col_indices: list[int] | None = None,
    save_path: str | None = "Transformat_Dists.png",
) -> None:
    """Show selected ratio distributions before and after PowerTransformer."""
    transformer = PowerTransformer()
    transformed = pd.DataFrame(
        transformer.fit_transform(original), columns=original.columns
    )
    if col_indices is None:
        col_indices = list(range(len(original.columns)))

    n = len(col_indices)
    fig, axes = plt.subplots(n, 2, figsize=(13, 4 * n))

    for row, idx in enumerate(col_indices):
        col_name = original.columns[idx]
        # Before
        axes[row, 0].hist(original[col_name], density=True, bins=30)
        axes[row, 0].set_xlabel(col_name)
        axes[row, 0].set_ylabel("Probability")
        axes[row, 0].grid(True)
        if row == 0:
            axes[row, 0].set_title("Before Transformation", fontsize=17)
        # After
        axes[row, 1].hist(transformed[col_name], density=True, bins=30)
        axes[row, 1].set_xlabel(col_name)
        axes[row, 1].set_ylabel("Probability")
        axes[row, 1].grid(True)
        if row == 0:
            axes[row, 1].set_title("After Transformation", fontsize=17)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300)

plot_before_after_transform(X, col_indices=[4, 6, 9, 10, 11])

# X Data for Final Stock Selection 2024/2023
Requires SimFin PROBulk Download

In [ ]:
x_2024_raw = pd.read_csv(FUNDAMENTALS_2024_FILE, index_col=0)

# Fix Net Income column name (verified against annual reports)
if "Net Income_x" in x_2024_raw.columns:
    x_2024_raw["Net Income (Common)"] = x_2024_raw["Net Income_x"]

X_2024 = build_ratio_features(x_2024_raw)
X_2024.to_csv(OUTPUT_RATIOS_2024_FILE)
print(f"Saved {OUTPUT_RATIOS_2024_FILE}  |  shape: {X_2024.shape}")

In [ ]:
X_2024